<a href="https://colab.research.google.com/github/HoniTahina/-arene-des-algos-Tahina-HONI-RIKA/blob/main/J4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Phase 0 — Installation & imports
!pip install -q flask flask-ngrok pyngrok streamlit joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                              f1_score, accuracy_score, roc_auc_score)
import joblib
import time

# Chargement du dataset
data = load_breast_cancer()
X, y = data.data, data.target

print(f"Dataset : {X.shape[0]} lignes, {X.shape[1]} features")
print(f"Classes : {data.target_names}")
print(f"Répartition : {np.bincount(y)} → {np.bincount(y)/len(y)*100}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 40.6 MB/s eta 0:00:00
Dataset : 569 lignes, 30 features
Classes : ['malignant' 'benign']
Répartition : [212 357] → [37.25834798 62.74165202]


In [2]:
# Phase 1 — Fonction de split

def split_train_val_test(X, y, test_size=0.2, val_size=0.2, random_state=42):
    """
    Découpe X, y en trois jeux : train, validation, test.
    Renvoie 6 objets : X_train, X_val, X_test, y_train, y_val, y_test.
    Les proportions sont respectées avec stratify=y.
    """
    if val_size <= 0 or val_size >= 1:
        raise ValueError(f"val_size doit être entre 0 et 1 (reçu : {val_size})")
    if test_size <= 0 or test_size >= 1:
        raise ValueError(f"test_size doit être entre 0 et 1 (reçu : {test_size})")

    # Premier split : isoler le test
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # Recalculer la proportion val sur le reste
    val_size_adjusted = val_size / (1 - test_size)

    # Deuxième split : isoler la validation
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_size_adjusted, random_state=random_state, stratify=y_temp
    )

    return X_train, X_val, X_test, y_train, y_val, y_test


# --- Test normal ---
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(X, y)
print(f"Train : {len(X_train)} | Validation : {len(X_val)} | Test : {len(X_test)}")
print(f"Total : {len(X_train)+len(X_val)+len(X_test)} == {len(X)} → {len(X_train)+len(X_val)+len(X_test)==len(X)}")

def check_stratification(y_full, y_train, y_val, y_test):
    ref = np.bincount(y_full) / len(y_full)
    for name, yy in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
        ratio = np.bincount(yy) / len(yy)
        ok = np.allclose(ref, ratio, atol=0.05)
        print(f"  {name} → {ratio.round(2)} {'✓' if ok else '✗'}")

print("Répartition des classes conservée dans chaque jeu :")
check_stratification(y, y_train, y_val, y_test)

# --- Test cas limite : val_size=0 ---
print("\n--- Cas limite : val_size=0 ---")
try:
    split_train_val_test(X, y, val_size=0)
    print("PROBLÈME : la fonction n'a pas planté !")
except ValueError as e:
    print(f"Erreur propre capturée : {e}")

# --- Test cas adversarial : dataset déséquilibré 95/5 ---
print("\n--- Cas adversarial : dataset 95/5 ---")
np.random.seed(42)
n = 500
X_imb = np.random.randn(n, 5)
y_imb = np.array([0]*475 + [1]*25)

Xtr, Xv, Xte, ytr, yv, yte = split_train_val_test(X_imb, y_imb)
print("Répartition dans chaque jeu (doit rester ~95/5) :")
check_stratification(y_imb, ytr, yv, yte)

Train : 341 | Validation : 114 | Test : 114
Total : 569 == 569 → True
Répartition des classes conservée dans chaque jeu :
  Train → [0.37 0.63] ✓
  Val → [0.38 0.62] ✓
  Test → [0.37 0.63] ✓

--- Cas limite : val_size=0 ---
Erreur propre capturée : val_size doit être entre 0 et 1 (reçu : 0)

--- Cas adversarial : dataset 95/5 ---
Répartition dans chaque jeu (doit rester ~95/5) :
  Train → [0.95 0.05] ✓
  Val → [0.95 0.05] ✓
  Test → [0.95 0.05] ✓
